# AXI test 4, Part 2 only (resume version): Qwen2.5-14B, 30 requests in a row
This is the **same test** as `axi_14b_test.ipynb` (same model, same 6 and 4 episodes, same seeds, same three policies, same reading rules, written beforehand in `docs/SCALE_TEST_14B.md`). Nothing is smaller. The difference: **progress is saved to your Google Drive after every finished episode**, so when Colab stops you (usage limit, disconnect), you do not lose it. Part 1 (Cells 6 and 7 of the first notebook) is done; you do not run it again.

* **Cell 6, rule stated:** 6 episodes, about 4 minutes each (about 25 minutes).
* **Cell 7, no rule:** 4 episodes, about 2.5 minutes each (about 10 minutes). It also prints the reading-rule lines.

**Before you start:** Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save. Run each cell **one at a time, top to bottom**, wait for the green tick before the next one. Cell 4 asks to connect Google Drive: allow it.

**If Colab stops you in the middle:** nothing is lost. Later (when Colab lets you back in, often the next day) open this notebook again and run **Cells 1 to 7 again, in order, even the ones that finished before**. Cell 4 reconnects Drive, Cell 6 prints `restored` and continues from the last finished episode, and a cell that was already finished ends in a few seconds.

Loading the model (Cell 5) costs about 12 minutes of GPU time each session, so start it when you have time.

### Cell 1: check the GPU (it must say Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: upload the code. A button 'Choose files' appears: pick `axi-colab-14b-v2.zip`

In [ ]:
from google.colab import files
up = files.upload()
name = list(up)[0]
!rm -rf /content/axi && mkdir -p /content/axi && unzip -q -o "{name}" -d /content/axi
!ls /content/axi/python

### Cell 3: install the 4-bit tools (about 1 minute)

In [ ]:
!pip -q install bitsandbytes accelerate

### Cell 4: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress is kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_14b")

### Cell 5: load the model (downloads about 30 GB each new session; about 10 to 15 minutes). If it stops with a red error about memory or disk, send me the red text

In [ ]:
import json, time, shutil, torch
print("free disk (GB):", round(shutil.disk_usage("/content").free / 1e9, 0), "(needs about 35)")
from colab.colab_llm_experiment import build_hf_llm
from axi.experiments.long_horizon import run_long, print_long
from axi.experiments.scope_test import print_exact, print_verdict
from axi.experiments.bigger_model import verdict_big_long

MODEL = "Qwen/Qwen2.5-14B-Instruct"
llm, model = build_hf_llm(MODEL, "cuda", load_4bit=True)
T0 = time.time()
def prog(i, n): print(f"  episode {i}/{n} done   {time.time() - T0:6.0f} s   (saved)" if store.persistent else f"  episode {i}/{n} done   {time.time() - T0:6.0f} s   (NOT saved to Drive)")
def save(res, path):
    json.dump(res, open(path, "w"), indent=1); return path
print("GPU memory used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

### Cell 6: PART 2, rule stated. 6 episodes x 30 requests, 3 policies (about 25 minutes). If this is a new session it says `restored` and continues

In [ ]:
T0 = time.time()
name = "axi_14b_long_rule"
print("progress file:", store.restore(name + ".jsonl"))
res_l_rule = run_long(llm, "state", 6, 30, 7, True, store.path(name + ".jsonl"), meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=store.progress(name + ".jsonl", prog), mix="scope")
print()
print_long(res_l_rule["summary"], "PART 2, rule stated, 14B")
print_exact(res_l_rule["summary"], "PART 2, rule stated, 14B")
files.download(save(res_l_rule, store.path(name + ".json")))
store.persist(name + ".json")

### Cell 7: PART 2, no rule. 4 episodes x 30 requests (about 10 minutes). Prints the reading-rule lines at the end

In [ ]:
T0 = time.time()
name = "axi_14b_long_norule"
print("progress file:", store.restore(name + ".jsonl"))
res_l_norule = run_long(llm, "state", 4, 30, 7, False, store.path(name + ".jsonl"), meta=dict(model=MODEL), policies=("blind", "repair_m6", "scope_m7"), progress=store.progress(name + ".jsonl", prog), mix="scope")
print()
print_long(res_l_norule["summary"], "PART 2, no rule, 14B")
print_exact(res_l_norule["summary"], "PART 2, no rule, 14B")
print_verdict(verdict_big_long(res_l_rule["summary"], res_l_norule["summary"]), "PART 2, both regimes")
files.download(save(res_l_norule, store.path(name + ".json")))
store.persist(name + ".json")

### Done
Files downloaded: `axi_14b_long_rule.json` and `axi_14b_long_norule.json`. Upload them to the chat (with `axi_14b_single_rule.json` and `axi_14b_single_norule.json` from the first notebook, if you have not sent those yet), with a screenshot of the printed tables and the 'reading rule' lines. Do not paste any keys anywhere.